# 2) Interactive Adapter Training (Colab)

Widget-driven continual adapter training with live progress and OOD calibration.

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch


def is_repo_root(path: Path) -> bool:
    return (path / "src").is_dir() and (path / "config").is_dir() and (path / "scripts").is_dir()


def maybe_clone_repo() -> Path | None:
    if os.environ.get("AADS_DISABLE_AUTO_CLONE") == "1":
        return None

    repo_url = os.environ.get("AADS_REPO_URL", "https://github.com/EfeErim/bitirmeprojesi.git")
    clone_target = Path(os.environ.get("AADS_REPO_CLONE_TARGET", "/content/bitirmeprojesi")).expanduser()

    if is_repo_root(clone_target):
        return clone_target

    if clone_target.exists() and any(clone_target.iterdir()):
        for child in clone_target.iterdir():
            if child.is_dir() and is_repo_root(child):
                return child
        return None

    clone_target.parent.mkdir(parents=True, exist_ok=True)
    print(f"Repository not found locally. Auto-cloning from: {repo_url}")
    completed = subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(clone_target)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)

    if completed.returncode == 0 and is_repo_root(clone_target):
        return clone_target
    return None


def resolve_repo_root() -> Path:
    env_candidates = [os.environ.get("AADS_REPO_ROOT"), os.environ.get("REPO_ROOT")]
    for raw in env_candidates:
        if not raw:
            continue
        p = Path(raw).expanduser().resolve()
        if is_repo_root(p):
            return p

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if is_repo_root(p):
            return p

    common = [
        Path("/content/bitirme projesi"),
        Path("/content/bitirmeprojesi"),
        Path("/content/aads_ulora"),
        Path("/content/drive/MyDrive/bitirme projesi"),
        Path("/content/drive/MyDrive/bitirmeprojesi"),
    ]
    for p in common:
        if is_repo_root(p):
            return p

    auto_cloned = maybe_clone_repo()
    if auto_cloned is not None:
        return auto_cloned

    raise FileNotFoundError("Repository root not found and auto-clone failed. Set AADS_REPO_ROOT, or set AADS_REPO_URL/AADS_REPO_CLONE_TARGET.")


try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

req = ROOT / "colab_notebooks" / "requirements_colab.txt"
if req.exists() and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = "cpu"

print(f"Repository root: {ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

BASE_CONFIG = json.loads((ROOT / "config" / "base.json").read_text(encoding="utf-8"))
TRAIN_CFG = BASE_CONFIG.get("training", {}).get("continual", {})
ADAPTER_CFG = TRAIN_CFG.get("adapter", {})
OOD_CFG = TRAIN_CFG.get("ood", {})

STATE = {
    "validated": False,
    "dataset_summary": None,
    "class_names": [],
    "runtime_dataset_root": None,
    "adapter": None,
    "loaders": None,
    "history": None,
    "calibration": None,
}

dataset_root_text = widgets.Text(value="data/class_root_dataset", description="Dataset:", layout=widgets.Layout(width="700px"))
crop_name_text = widgets.Text(value="tomato", description="Crop:")
epochs_slider = widgets.IntSlider(value=int(TRAIN_CFG.get("num_epochs", 1)), min=1, max=50, description="Epochs")
batch_size_slider = widgets.IntSlider(value=int(TRAIN_CFG.get("batch_size", 8)), min=1, max=64, description="Batch")
lr_slider = widgets.FloatLogSlider(value=float(TRAIN_CFG.get("learning_rate", 1e-4)), base=10, min=-6, max=-2, step=0.05, description="LR")
lora_r_slider = widgets.IntSlider(value=int(ADAPTER_CFG.get("lora_r", 16)), min=4, max=64, description="LoRA r")
lora_alpha_slider = widgets.IntSlider(value=int(ADAPTER_CFG.get("lora_alpha", 32)), min=8, max=128, description="LoRA α")
lora_dropout_slider = widgets.FloatSlider(value=float(ADAPTER_CFG.get("lora_dropout", 0.1)), min=0.0, max=0.5, step=0.01, description="Dropout")
ood_slider = widgets.FloatSlider(value=float(OOD_CFG.get("threshold_factor", 2.0)), min=1.0, max=5.0, step=0.1, description="OOD x")
validate_btn = widgets.Button(description="Validate & Start Training", button_style="success")
status_out = widgets.Output()
guide_html = widgets.HTML(value=(
    "<b>Parameter Guide (Suggested Values)</b><br>"
    "<ul style='margin-top:6px'>"
    "<li><b>Dataset</b>: class-root folder like <code>data/class_root_dataset</code> (required).</li>"
    "<li><b>Crop</b>: crop adapter name (suggested: <code>tomato</code>, <code>pepper</code>, <code>corn</code>).</li>"
    "<li><b>Epochs</b>: total training passes (start with <b>3-10</b>, increase if underfitting).</li>"
    "<li><b>Batch</b>: images per step (suggested <b>8</b> on T4/16GB, <b>16-32</b> on larger GPUs).</li>"
    "<li><b>LR</b>: learning rate (safe default <b>1e-4</b>; try <b>5e-5</b> if unstable).</li>"
    "<li><b>LoRA r</b>: adapter rank (suggested <b>16</b>; try <b>8</b> for speed, <b>32</b> for capacity).</li>"
    "<li><b>LoRA α</b>: scaling strength (suggested <b>32</b>; often about 2x <code>r</code>).</li>"
    "<li><b>Dropout</b>: LoRA regularization (suggested <b>0.1</b>; use <b>0.05-0.2</b>).</li>"
    "<li><b>OOD x</b>: OOD threshold factor (default <b>2.0</b>; increase to reduce false OOD).</li>"
    "</ul>"
    "Click <i>Validate & Start Training</i>, then run Cell 3 for dataset checks."
))


def gather_widget_config():
    return {
        "dataset_root": dataset_root_text.value.strip(),
        "crop_name": crop_name_text.value.strip(),
        "num_epochs": int(epochs_slider.value),
        "batch_size": int(batch_size_slider.value),
        "learning_rate": float(lr_slider.value),
        "lora_r": int(lora_r_slider.value),
        "lora_alpha": int(lora_alpha_slider.value),
        "lora_dropout": float(lora_dropout_slider.value),
        "ood_threshold_factor": float(ood_slider.value),
    }


def _on_validate_clicked(_):
    cfg = gather_widget_config()
    STATE["widget_config"] = cfg
    with status_out:
        status_out.clear_output()
        print("Config captured. Run Cell 3 for full dataset validation details.")


validate_btn.on_click(_on_validate_clicked)

panel = widgets.VBox([
    guide_html,
    dataset_root_text,
    crop_name_text,
    widgets.HBox([epochs_slider, batch_size_slider]),
    widgets.HBox([lr_slider, lora_r_slider, lora_alpha_slider]),
    widgets.HBox([lora_dropout_slider, ood_slider]),
    validate_btn,
    status_out,
])

display(panel)

In [ ]:
from scripts.evaluate_dataset_layout import evaluate_layout

cfg = STATE.get("widget_config") or {
    "dataset_root": dataset_root_text.value.strip(),
    "crop_name": crop_name_text.value.strip(),
}

raw_root = Path(cfg["dataset_root"]).expanduser()
if not raw_root.is_absolute():
    raw_root = (ROOT / raw_root).resolve()

summary = evaluate_layout(raw_root)
STATE["dataset_summary"] = summary

print("Dataset root:", raw_root)
print("Layout ok:", summary.get("ok"))
print("Summary:", summary.get("summary", {}))

if summary.get("errors"):
    print("Errors:")
    for item in summary["errors"]:
        print(" -", item)

if summary.get("warnings"):
    print("Warnings:")
    for item in summary["warnings"]:
        print(" -", item)

class_names = [entry.get("class_name") for entry in summary.get("classes", []) if entry.get("class_name")]
STATE["class_names"] = class_names
print("Discovered classes:", class_names)

STATE["validated"] = bool(summary.get("ok")) and len(class_names) > 0

In [ ]:
from src.adapter.independent_crop_adapter import IndependentCropAdapter
from src.utils.data_loader import create_training_loaders


def prepare_runtime_dataset_layout(class_root: Path, crop_name: str, seed: int = 42) -> Path:
    runtime_root = ROOT / "data" / "runtime_notebook_datasets"
    crop_root = runtime_root / crop_name
    if crop_root.exists():
        shutil.rmtree(crop_root)

    random.seed(seed)
    splits = ["continual", "val", "test"]

    for cls_dir in sorted([p for p in class_root.iterdir() if p.is_dir()]):
        image_files = [p for p in cls_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}]
        random.shuffle(image_files)
        n = len(image_files)
        if n == 0:
            continue

        train_n = max(1, int(0.8 * n))
        val_n = max(1, int(0.1 * n)) if n >= 3 else 0
        if train_n + val_n >= n:
            val_n = max(0, n - train_n - 1)
        test_n = max(0, n - train_n - val_n)

        split_map = {
            "continual": image_files[:train_n],
            "val": image_files[train_n:train_n + val_n],
            "test": image_files[train_n + val_n:train_n + val_n + test_n],
        }

        for split in splits:
            target = crop_root / split / cls_dir.name
            target.mkdir(parents=True, exist_ok=True)
            for src_path in split_map[split]:
                dst = target / src_path.name
                if not dst.exists():
                    shutil.copy2(src_path, dst)

    return runtime_root


if not STATE.get("validated"):
    raise RuntimeError("Run Cell 3 and ensure dataset validation passes first.")

cfg = STATE["widget_config"]
crop_name = cfg["crop_name"]
class_root = Path(cfg["dataset_root"]).expanduser()
if not class_root.is_absolute():
    class_root = (ROOT / class_root).resolve()

runtime_data_root = prepare_runtime_dataset_layout(class_root=class_root, crop_name=crop_name)
STATE["runtime_dataset_root"] = runtime_data_root

continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
continual_cfg["backbone"]["model_name"] = continual_cfg.get("backbone", {}).get("model_name", "facebook/dinov3-giant")
continual_cfg["device"] = DEVICE
continual_cfg["num_epochs"] = int(cfg["num_epochs"])
continual_cfg["batch_size"] = int(cfg["batch_size"])
continual_cfg["learning_rate"] = float(cfg["learning_rate"])
continual_cfg["adapter"]["lora_r"] = int(cfg["lora_r"])
continual_cfg["adapter"]["lora_alpha"] = int(cfg["lora_alpha"])
continual_cfg["adapter"]["lora_dropout"] = float(cfg["lora_dropout"])
continual_cfg["ood"]["threshold_factor"] = float(cfg["ood_threshold_factor"])

adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})

loaders = create_training_loaders(
    data_dir=str(runtime_data_root),
    crop=crop_name,
    batch_size=int(cfg["batch_size"]),
    num_workers=0,
    use_cache=False,
)

STATE["adapter"] = adapter
STATE["loaders"] = loaders
STATE["continual_config"] = continual_cfg

print("Engine initialized.")
print("Runtime dataset root:", runtime_data_root)
print("Classes:", len(STATE["class_names"]))
print("Resolved LoRA target modules:", len(adapter.target_modules_resolved))
print("Fusion layers:", continual_cfg.get("fusion", {}).get("layers"))
print("Fusion output dim:", continual_cfg.get("fusion", {}).get("output_dim"))

trainer = adapter._trainer
if trainer is not None:
    total_params = 0
    trainable_params = 0
    for p in trainer.adapter_model.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    for p in trainer.classifier.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    for p in trainer.fusion.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()

    print(f"Total params (adapter+heads): {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

In [ ]:
from IPython.display import clear_output, display

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
loaders = STATE["loaders"]
num_epochs = int(STATE["widget_config"]["num_epochs"])

progress_bar = widgets.IntProgress(value=0, min=0, max=max(1, len(loaders["train"])), description="Batch")
status_html = widgets.HTML(value="<b>Ready.</b>")
loss_out = widgets.Output()
display(progress_bar, status_html, loss_out)

train_loss_curve = []
start_time = time.time()


def progress_cb(event):
    if "batch" in event:
        progress_bar.max = max(1, int(event.get("total_batches", 1)))
        progress_bar.value = int(event.get("batch", 0))

        elapsed = time.time() - start_time
        batches_done = max(1, ((event.get("epoch", 1) - 1) * progress_bar.max) + event.get("batch", 1))
        total_batches = max(1, num_epochs * progress_bar.max)
        eta = (elapsed / batches_done) * max(0, total_batches - batches_done)

        status_html.value = (
            f"<b>Epoch:</b> {event.get('epoch', 0)} | "
            f"<b>Batch:</b> {event.get('batch', 0)}/{event.get('total_batches', 0)} | "
            f"<b>Batch Loss:</b> {event.get('batch_loss', 0.0):.4f} | "
            f"<b>Elapsed:</b> {elapsed:.1f}s | <b>ETA:</b> {eta:.1f}s"
        )

    if "epoch_done" in event:
        train_loss_curve.append(float(event["epoch_loss"]))
        with loss_out:
            clear_output(wait=True)
            plt.figure(figsize=(6, 3))
            plt.plot(range(1, len(train_loss_curve) + 1), train_loss_curve, marker="o")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title("Training Loss")
            plt.grid(True, alpha=0.3)
            plt.show()


train_result = adapter.train_increment(
    loaders["train"],
    num_epochs=num_epochs,
    progress_callback=progress_cb,
)

elapsed_total = time.time() - start_time
STATE["history"] = train_result.get("history", {})

print("Training complete")
print("Total time (s):", round(elapsed_total, 2))
print("History:", STATE["history"])

In [ ]:
if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")

if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty; cannot calibrate OOD.")

cal = adapter.calibrate_ood(val_loader)
STATE["calibration"] = cal

num_classes = cal.get("ood_calibration", {}).get("num_classes", 0)
version = cal.get("ood_calibration", {}).get("version", 0)
status_color = "green" if int(num_classes) > 0 else "red"

widgets.HTML(
    value=f"<b style='color:{status_color}'>OOD calibration completed</b> | classes={num_classes} | version={version}"
)

In [ ]:
if STATE.get("adapter") is None:
    raise RuntimeError("Run Cell 4 first.")

checkpoint_dir = ROOT / "outputs" / "colab_notebook_training"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

STATE["adapter"].save_adapter(str(checkpoint_dir))
asset_dir = checkpoint_dir / "continual_sd_lora_adapter"

print("Saved adapter directory:", asset_dir)
if not asset_dir.exists():
    raise RuntimeError(f"Expected adapter dir missing: {asset_dir}")

for p in sorted(asset_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(ROOT))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")
if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty.")

trainer = adapter._trainer
if trainer is None:
    raise RuntimeError("Trainer not initialized.")

trainer.adapter_model.eval()
trainer.classifier.eval()
trainer.fusion.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in val_loader:
        images = batch["images"].to(trainer.device)
        labels = batch["labels"].to(trainer.device)
        logits = trainer.forward_logits(images)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

if not y_true:
    raise RuntimeError("No validation samples available.")

classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda kv: kv[1])]
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()